In [2]:
import os
import h5py
import numpy as np
import pandas as pd

In [3]:
# Read in labeled folder names and timestamps
Cluster_detail_results = pd.read_csv( os.path.join(r'Y:\3darena_behavior\mitopark\mitopark_042025\042025_2mp\week8-10-12-14-23Comp_video\Combined_Results\Results\test1\Cluster_detail_results.csv') )

In [4]:
# CHANGE HERE: path to your mat file
filename = r"Y:\3darena_behavior\mitopark\mitopark_042025\042025_2mp\week8-10-12-14-23Comp_video\Combined_Results\Results\test1\session_1_out.mat"

In [5]:
# Read in values from clustersIdx, set automatically to the cluster value it represents rather than the actual value of the cell
if os.path.exists(filename):
    with h5py.File(filename, 'r') as file:
        library = file['Library']
        clustersIdx = library['clustersIdx']
        
        group_labels = []
        for i in range(clustersIdx.shape[0]):
            idxref = clustersIdx[i, 0]
            cluster_indices = np.array(file[idxref]).flatten()
            group_labels.extend([i+1] * len(cluster_indices))
        
        group_labels_df = pd.DataFrame(group_labels, columns=['GroupLabel'])
        print(f"Group labels shape: {group_labels_df.shape}")
else:
    raise FileNotFoundError('Could not find session_1_out.mat in dedicated subdirectory')


group_labels_df

Group labels shape: (121067, 1)


,GroupLabel
0,1
1,1
2,1
3,1
4,1
...,...
121062,97
121063,97
121064,97
121065,97


In [6]:
# Read in the feature values from clusters and stack on top of each other
if os.path.exists(filename):
    print('Loading session file from expected Results/test1 directory')
    with h5py.File(filename, 'r') as file:
        if 'Library' not in file:
            raise KeyError("The key 'Library' was not found in the session file.")
        library = file['Library']
        clusters = library['clusters']
        
        features_data = []
        
        for i in range(clusters.shape[0]):
            cref = clusters[i, 0]
            feature_data = np.array(file[cref]).T 
            features_data.append(feature_data)
        
        features_matrix = np.vstack(features_data)
        features_df = pd.DataFrame(features_matrix)
        print(f"Loaded {len(features_data)} clusters with total points: {features_matrix.shape[0]}")
        print(f"Combined matrix shape: {features_matrix.shape}")
else:
    raise FileNotFoundError('Could not find session_1_out.mat in dedicated subdirectory')
features_df['cluster'] = group_labels_df['GroupLabel'].values


features_df

Loading session file from expected Results/test1 directory
Loaded 97 clusters with total points: 121067
Combined matrix shape: (121067, 30)


,0,1,2,3,4,5,6,7,8,9,...,21,22,23,24,25,26,27,28,29,cluster
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.016667,0.983333,0.000000,...,0.0,0.116667,0.150000,0.366667,0.366667,0.000000,0.000000,0.000000,1.000000,1
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.033333,0.966667,...,0.0,0.066667,0.066667,0.316667,0.300000,0.250000,0.000000,0.016667,0.983333,1
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,1.000000,...,0.0,0.000000,0.233333,0.350000,0.283333,0.066667,0.066667,0.000000,1.000000,1
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,1.000000,...,0.0,0.000000,0.100000,0.400000,0.350000,0.150000,0.000000,0.000000,1.000000,1
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,1.000000,...,0.0,0.066667,0.133333,0.266667,0.416667,0.100000,0.016667,0.000000,1.000000,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121062,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.633333,0.366667,0.000000,0.000000,1.000000,0.000000,97
121063,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.450000,0.550000,0.000000,0.000000,1.000000,0.000000,97
121064,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.366667,0.633333,0.000000,0.000000,1.000000,0.000000,97
121065,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.333333,0.666667,0.000000,0.000000,1.000000,0.000000,97


In [7]:
# Create combined matrix by matching first occurrence of each cluster in features_df to first occurrence in Cluster_detail_results, keeping order of Cluster_detail_results
def match_features_to_details(features_df, detail_df):
    matched_rows = []
    cluster_counts = {}
    for idx, detail_row in detail_df.iterrows():
        cluster_val = detail_row['ClusterIdx']
        count = cluster_counts.get(cluster_val, 0)
        feature_rows = features_df[features_df['cluster'] == cluster_val]
        if count < len(feature_rows):
            feature_row = feature_rows.iloc[count]
            combined_row = feature_row.to_dict()
            combined_row['Timestamp'] = detail_row['Timestamp'] if 'Timestamp' in detail_row else None
            combined_row['ClusterIdx'] = cluster_val
            combined_row['Folder_Name'] = detail_row['Folder_Name'] if 'Folder_Name' in detail_row else None
            matched_rows.append(combined_row)
            cluster_counts[cluster_val] = count + 1
        else:
            combined_row = {col: None for col in features_df.columns}
            combined_row['Timestamp'] = detail_row['Timestamp'] if 'Timestamp' in detail_row else None
            combined_row['ClusterIdx'] = cluster_val
            combined_row['Folder_Name'] = detail_row['Folder_Name'] if 'Folder_Name' in detail_row else None
            matched_rows.append(combined_row)
    return pd.DataFrame(matched_rows)

In [8]:
combined_matrix = match_features_to_details(features_df, Cluster_detail_results)
combined_matrix

,0,1,2,3,4,5,6,7,8,9,...,24,25,26,27,28,29,cluster,Timestamp,ClusterIdx,Folder_Name
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.016667,0.983333,0.000000,...,0.366667,0.366667,0.000000,0.000000,0.000000,1.000000,1.0,379.3972,1,1_week8_1
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.033333,0.966667,...,0.316667,0.300000,0.250000,0.000000,0.016667,0.983333,1.0,379.6969,1,1_week8_1
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,1.000000,...,0.350000,0.283333,0.066667,0.066667,0.000000,1.000000,1.0,379.9966,1,1_week8_1
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,1.000000,...,0.400000,0.350000,0.150000,0.000000,0.000000,1.000000,1.0,380.2964,1,1_week8_1
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,1.000000,...,0.266667,0.416667,0.100000,0.016667,0.000000,1.000000,1.0,380.5962,1,1_week8_1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121062,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,1.000000,...,0.483333,0.516667,0.000000,0.000000,0.816667,0.183333,96.0,42243.7645,96,7_week23
121063,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,1.000000,...,0.583333,0.416667,0.000000,0.000000,1.000000,0.000000,91.0,42244.1195,91,7_week23
121064,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,1.000000,...,0.633333,0.366667,0.000000,0.000000,0.850000,0.150000,96.0,42244.4695,96,7_week23
121065,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.283333,0.716667,...,0.550000,0.450000,0.000000,0.000000,0.633333,0.366667,79.0,42244.8145,79,7_week23


In [9]:
def get_week_number(folder_num):
    if folder_num in ["1_week8_1", "2_week8_2"]:
        return 8
    elif folder_num in ["3_week10_1", "4_week10_2"]:
        return 10
    elif folder_num in ["5_week12"]:
        return 12
    elif folder_num in ["6_week14"]:
        return 14
    elif folder_num in ["7_week23"]:
        return 23

combined_matrix['Week_Number'] = combined_matrix['Folder_Name'].apply(get_week_number)
combined_matrix['Week_Number'] = combined_matrix['Week_Number'].fillna(0).astype(int)
combined_matrix = combined_matrix[combined_matrix['Week_Number'] != 0]
combined_matrix

,0,1,2,3,4,5,6,7,8,9,...,25,26,27,28,29,cluster,Timestamp,ClusterIdx,Folder_Name,Week_Number
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.016667,0.983333,0.000000,...,0.366667,0.000000,0.000000,0.000000,1.000000,1.0,379.3972,1,1_week8_1,8
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.033333,0.966667,...,0.300000,0.250000,0.000000,0.016667,0.983333,1.0,379.6969,1,1_week8_1,8
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,1.000000,...,0.283333,0.066667,0.066667,0.000000,1.000000,1.0,379.9966,1,1_week8_1,8
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,1.000000,...,0.350000,0.150000,0.000000,0.000000,1.000000,1.0,380.2964,1,1_week8_1,8
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,1.000000,...,0.416667,0.100000,0.016667,0.000000,1.000000,1.0,380.5962,1,1_week8_1,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121061,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,1.000000,...,0.383333,0.083333,0.000000,0.166667,0.833333,18.0,42243.4245,18,7_week23,23
121062,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,1.000000,...,0.516667,0.000000,0.000000,0.816667,0.183333,96.0,42243.7645,96,7_week23,23
121063,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,1.000000,...,0.416667,0.000000,0.000000,1.000000,0.000000,91.0,42244.1195,91,7_week23,23
121064,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,1.000000,...,0.366667,0.000000,0.000000,0.850000,0.150000,96.0,42244.4695,96,7_week23,23


In [10]:
# Clean up the finalized combined df (Column Titles : "Feature 1" ... "Feature 30" , "Timestamp" , "Cluster", "Week_Number")
feature_cols = {i: f'Feature{i+1}' for i in range(30)}
combined_matrix = combined_matrix.rename(columns=feature_cols)
combined_matrix = combined_matrix.drop(columns=['cluster'])
combined_matrix = combined_matrix.drop(columns=['Folder_Name'])
combined_matrix = combined_matrix.rename(columns={'ClusterIdx': 'Cluster'})
combined_matrix

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10,...,Feature24,Feature25,Feature26,Feature27,Feature28,Feature29,Feature30,Timestamp,Cluster,Week_Number
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.016667,0.983333,0.000000,...,0.150000,0.366667,0.366667,0.000000,0.000000,0.000000,1.000000,379.3972,1,8
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.033333,0.966667,...,0.066667,0.316667,0.300000,0.250000,0.000000,0.016667,0.983333,379.6969,1,8
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,1.000000,...,0.233333,0.350000,0.283333,0.066667,0.066667,0.000000,1.000000,379.9966,1,8
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,1.000000,...,0.100000,0.400000,0.350000,0.150000,0.000000,0.000000,1.000000,380.2964,1,8
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,1.000000,...,0.133333,0.266667,0.416667,0.100000,0.016667,0.000000,1.000000,380.5962,1,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121061,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,1.000000,...,0.066667,0.466667,0.383333,0.083333,0.000000,0.166667,0.833333,42243.4245,18,23
121062,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,1.000000,...,0.000000,0.483333,0.516667,0.000000,0.000000,0.816667,0.183333,42243.7645,96,23
121063,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,1.000000,...,0.000000,0.583333,0.416667,0.000000,0.000000,1.000000,0.000000,42244.1195,91,23
121064,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,1.000000,...,0.000000,0.633333,0.366667,0.000000,0.000000,0.850000,0.150000,42244.4695,96,23


In [11]:
# CHANGE HERE: File name
output_file = r"C:\Users\gangliagurdian\Desktop\Mias Folder\new graphs - AC 9152025\data\combined_matrix_final_total_mp.csv"
combined_matrix.to_csv(output_file, index=False)
print(f"Saved as {output_file}")
print(combined_matrix.head())

Saved as C:\Users\gangliagurdian\Desktop\Mias Folder\new graphs - AC 9152025\data\combined_matrix_final_total_mp.csv
   Feature1  Feature2  Feature3  Feature4  Feature5  Feature6  Feature7  \
0       0.0       0.0       0.0       0.0       0.0       0.0       0.0   
1       0.0       0.0       0.0       0.0       0.0       0.0       0.0   
2       0.0       0.0       0.0       0.0       0.0       0.0       0.0   
3       0.0       0.0       0.0       0.0       0.0       0.0       0.0   
4       0.0       0.0       0.0       0.0       0.0       0.0       0.0   

   Feature8  Feature9  Feature10  ...  Feature24  Feature25  Feature26  \
0  0.016667  0.983333   0.000000  ...   0.150000   0.366667   0.366667   
1  0.000000  0.033333   0.966667  ...   0.066667   0.316667   0.300000   
2  0.000000  0.000000   1.000000  ...   0.233333   0.350000   0.283333   
3  0.000000  0.000000   1.000000  ...   0.100000   0.400000   0.350000   
4  0.000000  0.000000   1.000000  ...   0.133333   0.266667   